# Regional (ROI) Dataframe

Aggregates plaque-level data from `data.parquet` to a per-subject × per-atlas-region
summary and saves the result as `roi_data.parquet`.

Each row represents one subject in one atlas region and contains:
- Plaque count and volume metrics
- Plaque diameter metrics
- Vessel-proximity fractions (inside / near / far)
- Signed-distance-transform (SDT) summary statistics
- Plaque density (`plaque_count / region_volume_mm3`)


In [1]:
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

# ── Analysis parameters (kept in sync with other notebooks) ──────────────────
VOL_THRESH_ML     = 1e-4          # discard implausibly large plaques
TAU_UM            = 5.0           # near-vessel boundary (µm)
VESSEL_BINS_UM    = [0, 10, 20, np.inf]
VESSEL_BIN_LABELS = ["<10 µm", "10\u201320 µm", ">20 µm"]

TREAT_ORDER = ["PBS", "Lecanemab"]
GENO_ORDER  = ["ApoE3", "ApoE4"]

GROUP_COLS = ["subject", "treatment", "genotype", "sex", "index", "name"]


## 1 · Load and filter plaque data


In [2]:
df_raw = pd.read_parquet("data.parquet")

df = df_raw.loc[df_raw["plaque_vol_ml"] <= VOL_THRESH_ML].copy()
df["treatment"] = pd.Categorical(df["treatment"], categories=TREAT_ORDER, ordered=True)
df["genotype"]  = pd.Categorical(df["genotype"],  categories=GENO_ORDER,  ordered=True)

print(f"Retained {len(df):,} plaques from {df['subject'].nunique()} subjects "
      f"(removed {len(df_raw) - len(df):,} oversized plaques)")
df.head(3)


Retained 1,157,491 plaques from 16 subjects (removed 81,933 oversized plaques)


,pos_x,pos_y,pos_z,nvoxels,template_x,template_y,template_z,stain,sdt_CD31,index,...,subject,participant_id,lightsheet_id,genotype,sex,treatment,sdt_CD31_um,plaque_vol_um3,plaque_vol_ml,equiv_diam_um
2,4.278144,3.600416,-0.864738,331.0,0.382313,-6.709099,4.048942,Abeta,0.011406,11046,...,sub-AS40F2,sub-AS40F2,a,ApoE4,F,PBS,11.405924,2330.24,0.000002,16.448792
3,4.300077,3.604410,-0.874487,875.0,0.416069,-6.709476,4.041669,Abeta,0.002918,11046,...,sub-AS40F2,sub-AS40F2,a,ApoE4,F,PBS,2.918425,6160.00,0.000006,22.743678
5,2.883424,3.775627,-0.875569,229.0,-1.703566,-6.508430,3.926726,Abeta,0.003027,0,...,sub-AS40F2,sub-AS40F2,a,ApoE4,F,PBS,3.027425,1612.16,0.000002,14.547996


## 2 · Load atlas look-up table


In [3]:
lut = pd.read_csv("tpl-ABAv3_seg-all_dseg.tsv", sep="\t")
print(f"LUT: {len(lut):,} regions")
lut.head()


LUT: 2,654 regions


,index,name,abbreviation,color,volume_mm3
0,0,left root,root-L,000000,701.337955
1,1,left Basic cell groups and regions,grey-L,BFDAE3,0.000000
2,2,left Cerebrum,CH-L,B0F0FF,0.000000
3,3,left Cerebral cortex,CTX-L,B0FFB8,0.000000
4,4,left Cortical plate,CTXpl-L,70FF70,0.000000


## 3 · Classify plaques by vessel proximity

Mirrors the `classify_plaques()` function in `vessel_spatial_analysis.ipynb`.


In [4]:
def classify_plaques(
    plaque_df,
    tau_um=TAU_UM,
    vessel_bins_um=VESSEL_BINS_UM,
    vessel_bin_labels=VESSEL_BIN_LABELS,
):
    """Add vessel-relation category and min-vessel-diameter estimate."""
    out = plaque_df.copy()
    out["vessel_relation"] = pd.cut(
        out["sdt_CD31_um"],
        bins=[-np.inf, 0, tau_um, np.inf],
        labels=[
            "inside vessel",
            f"near vessel (0\u2013{tau_um:g} \u00b5m)",
            f"far from vessel (>{tau_um:g} \u00b5m)",
        ],
        include_lowest=True,
        right=True,
    )
    out["min_vessel_diam_um"] = np.nan
    inside = out["sdt_CD31_um"] < 0
    out.loc[inside, "min_vessel_diam_um"] = (
        out.loc[inside, "equiv_diam_um"] - 2 * out.loc[inside, "sdt_CD31_um"]
    )
    out["vessel_diam_bin"] = pd.cut(
        out["min_vessel_diam_um"],
        bins=vessel_bins_um,
        labels=vessel_bin_labels,
        include_lowest=True,
    )
    return out


df = classify_plaques(df)
df["vessel_relation"].value_counts()


vessel_relation
inside vessel              622701
far from vessel (>5 µm)    325894
near vessel (0–5 µm)       208896
Name: count, dtype: int64

## 4 · Aggregate to per-subject × per-ROI summary

For each subject × atlas region we compute:
- **Plaque count** and **volume** (sum, mean, median)
- **Equivalent diameter** (mean, median)
- **Signed distance transform** to nearest vessel (mean, median)
- **Vessel-proximity fractions** (inside / near / far as a share of that ROI's plaques)


In [5]:
def build_roi_dataframe(plaque_df, group_cols=GROUP_COLS):
    """Aggregate plaque metrics per subject per atlas ROI."""
    grp = plaque_df.groupby(group_cols, observed=True)

    # ── Core metrics ─────────────────────────────────────────────────────────
    agg = grp.agg(
        plaque_count   =("nvoxels",        "size"),
        total_vol_ml   =("plaque_vol_ml",   "sum"),
        mean_vol_ml    =("plaque_vol_ml",   "mean"),
        median_vol_ml  =("plaque_vol_ml",   "median"),
        mean_diam_um   =("equiv_diam_um",   "mean"),
        median_diam_um =("equiv_diam_um",   "median"),
        total_vol_um3  =("plaque_vol_um3",  "sum"),
        mean_sdt_um    =("sdt_CD31_um",     "mean"),
        median_sdt_um  =("sdt_CD31_um",     "median"),
    ).reset_index()

    # ── Vessel-proximity fractions ────────────────────────────────────────────
    rel_counts = (
        plaque_df
        .groupby(group_cols + ["vessel_relation"], observed=True)
        .size()
        .rename("n_rel")
        .reset_index()
        .merge(agg[group_cols + ["plaque_count"]], on=group_cols)
    )
    rel_counts["frac"] = rel_counts["n_rel"] / rel_counts["plaque_count"]

    # Pivot to wide format so each category becomes its own column
    PROX_RENAME = {
        "inside vessel":                         "frac_inside_vessel",
        f"near vessel (0\u2013{TAU_UM:g} \u00b5m)": "frac_near_vessel",
        f"far from vessel (>{TAU_UM:g} \u00b5m)": "frac_far_vessel",
    }
    rel_wide = (
        rel_counts
        .pivot_table(index=group_cols, columns="vessel_relation", values="frac", fill_value=0.0)
        .rename(columns=PROX_RENAME)
        .reset_index()
    )
    rel_wide.columns.name = None

    return agg.merge(rel_wide, on=group_cols, how="left")


roi_df = build_roi_dataframe(df)
print(f"ROI dataframe: {len(roi_df):,} rows  "
      f"({roi_df['subject'].nunique()} subjects × up to {roi_df['index'].nunique()} regions)")
roi_df.head()


ROI dataframe: 13,724 rows  (16 subjects × up to 1280 regions)


,subject,treatment,genotype,sex,index,name,plaque_count,total_vol_ml,mean_vol_ml,median_vol_ml,mean_diam_um,median_diam_um,total_vol_um3,mean_sdt_um,median_sdt_um,frac_inside_vessel,frac_near_vessel,frac_far_vessel
0,sub-AS161F3,PBS,ApoE3,F,0,left root,30885,0.234113,0.000008,0.000003,21.147472,18.355968,2.341129e+08,0.050244,-2.887072,0.646301,0.160855,0.192844
1,sub-AS161F3,PBS,ApoE3,F,7,"left Frontal pole, layer 1",154,0.000696,0.000005,0.000002,18.519932,16.765627,6.955238e+05,1.772841,-2.563844,0.577922,0.136364,0.285714
2,sub-AS161F3,PBS,ApoE3,F,8,"left Frontal pole, layer 2/3",18,0.000048,0.000003,0.000002,16.680384,15.697833,4.809024e+04,-4.095661,-4.546432,0.944444,0.055556,0.000000
3,sub-AS161F3,PBS,ApoE3,F,9,"left Frontal pole, layer 5",1,0.000002,0.000002,0.000002,14.419813,14.419813,1.569920e+03,-4.433161,-4.433161,1.000000,0.000000,0.000000
4,sub-AS161F3,PBS,ApoE3,F,19,"left Primary motor area, Layer 1",464,0.003720,0.000008,0.000003,21.443076,18.663372,3.719683e+06,-1.217587,-3.509387,0.683190,0.142241,0.174569


## 5 · Merge region volumes and compute plaque density

**Density** = `plaque_count / region_volume_mm3`  
**Volume density** = `total_vol_ml / region_volume_mm3` (total plaque volume per unit brain volume)


In [6]:
roi_df = roi_df.merge(
    lut[["index", "volume_mm3"]],
    on="index",
    how="left",
)

roi_df["plaque_density"]  = roi_df["plaque_count"] / roi_df["volume_mm3"]
roi_df["vol_density_ml"]  = roi_df["total_vol_ml"] / roi_df["volume_mm3"]

print(f"Rows with valid region volume: "
      f"{roi_df['volume_mm3'].notna().sum():,} / {len(roi_df):,}")
roi_df[["subject", "name", "plaque_count", "volume_mm3",
         "plaque_density", "vol_density_ml"]].head()


Rows with valid region volume: 13,724 / 13,724


,subject,name,plaque_count,volume_mm3,plaque_density,vol_density_ml
0,sub-AS161F3,left root,30885,701.337955,44.037257,0.000334
1,sub-AS161F3,"left Frontal pole, layer 1",154,0.126750,1214.990080,0.005487
2,sub-AS161F3,"left Frontal pole, layer 2/3",18,0.117031,153.805066,0.000411
3,sub-AS161F3,"left Frontal pole, layer 5",1,0.185797,5.382222,0.000008
4,sub-AS161F3,"left Primary motor area, Layer 1",464,0.664609,698.154428,0.005597


## 6 · Save ROI dataframe


In [7]:
roi_df.to_parquet("roi_data.parquet")
print(f"Saved roi_data.parquet  —  "
      f"{len(roi_df):,} rows × {len(roi_df.columns)} columns")
roi_df.dtypes


Saved roi_data.parquet  —  13,724 rows × 21 columns


subject                    str
treatment             category
genotype              category
sex                        str
index                    int64
name                       str
plaque_count             int64
total_vol_ml           float64
mean_vol_ml            float64
median_vol_ml          float64
mean_diam_um           float64
median_diam_um         float64
total_vol_um3          float64
mean_sdt_um            float64
median_sdt_um          float64
frac_inside_vessel     float64
frac_near_vessel       float64
frac_far_vessel        float64
volume_mm3             float64
plaque_density         float64
vol_density_ml         float64
dtype: object